# 🚗 HUẤN LUYỆN CHUYÊN GIA RT-DETR (DEEP ENSEMBLES) - BDD100K

### 📌 Hướng dẫn dành cho thành viên nhóm:
1. **Bật GPU:** Nhìn sang góc phải màn hình Kaggle $\rightarrow$ Mục **Settings** $\rightarrow$ **Accelerator** $\rightarrow$ Chọn **GPU T4 x2** (hoặc GPU T4 trên Colab).
2. **Bật Internet:** Vào menu **Settings** $\rightarrow$ Bấm **Turn on internet** $\rightarrow$ Bấm xác nhận.
3. **Thêm Dữ liệu:** Bấm **"Add Input"** (góc trên bên phải) $\rightarrow$ Tìm kiếm `solesensei` $\rightarrow$ Thêm bộ dữ liệu **BDD100K** vào Notebook.
4. **Chỉnh sửa Cell 2:** Đổi giá trị `ENSEMBLE_ID` theo số thứ tự của bạn (1, 2, 3, 4, hoặc 5).
5. **Treo máy chạy ngầm:** Bấm nút **Save Version** góc trên bên phải $\rightarrow$ Chọn **Save & Run All (Commit)** $\rightarrow$ Bấm **Save**.
6. **Tải kết quả:** Khi chạy xong, nhìn sang cột **Output** bên phải $\rightarrow$ Tải file `expert_X.pth` về nộp lại cho nhóm trưởng!

In [ ]:
# ==============================================================================
# CELL 1: CÀI ĐẶT CÁC THƯ VIỆN CẦN THIẾT
# ==============================================================================
!pip install -q "transformers>=4.38.0" timm accelerate
print("[*] Đã hoàn tất cài đặt thư viện!")
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

[*] Đã hoàn tất cài đặt thư viện!


In [ ]:
# ==============================================================================
# CELL 2: CẤU HÌNH THÀNH VIÊN VÀ SEED NGẪU NHIÊN CHUẨN BÀI BÁO
# ==============================================================================
# >>> [BẠN CHỈ CẦN SỬA DUY NHẤT DÒNG NÀY (TỪ 1 ĐẾN 5)] <<<
ENSEMBLE_ID = 1
# ========================================================

import random
import numpy as np
import torch

def set_seed(ensemble_id):
    # Model 1 = 42, Model 2 = 142, Model 3 = 242, Model 4 = 342, Model 5 = 442
    seed = 42 + ((ensemble_id - 1) * 100)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    print(f"[*] ĐÃ THIẾT LẬP THÀNH CÔNG: Chuyên gia số {ensemble_id} với SEED = {seed}")
    return seed

CURRENT_SEED = set_seed(ENSEMBLE_ID)

In [ ]:
# ==============================================================================
# CELL 3: BỘ ĐỌC DỮ LIỆU CHUẨN XÁC THEO CẤU TRÚC KAGGLE
# ==============================================================================
import os
import json
from PIL import Image
from torch.utils.data import Dataset

BASE_DATA_DIR = "/kaggle/input/datasets/solesensei/solesensei_bdd100k"
TRAIN_JSON = os.path.join(BASE_DATA_DIR, "bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_train.json")
VAL_JSON = os.path.join(BASE_DATA_DIR, "bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_val.json")
TRAIN_IMG_DIR = os.path.join(BASE_DATA_DIR, "bdd100k/bdd100k/images/100k/train")
VAL_IMG_DIR = os.path.join(BASE_DATA_DIR, "bdd100k/bdd100k/images/100k/val")

TARGET_CLASS = 'car'
COCO_CAR_ID = 3

class BDD100kDataset(Dataset):
    def __init__(self, json_path, img_dir, split_name="TRAIN", processor=None):
        self.processor = processor
        
        print(f"[*] Đang quét danh sách ảnh trên ổ đĩa từ: {img_dir}...")
        self.img_map = {}
        for r, _, files in os.walk(img_dir):
            for f in files:
                if f.endswith('.jpg'):
                    self.img_map[f] = os.path.join(r, f)
        print(f"[*] Tìm thấy {len(self.img_map)} ảnh trên ổ đĩa cho tập [{split_name}].")
        
        print(f"[*] Đang nạp file nhãn: {json_path}...")
        with open(json_path, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)
            
        self.samples = []
        for item in raw_data:
            img_name = item['name']
            actual_path = self.img_map.get(img_name) or self.img_map.get(f"{img_name}.jpg")
            if actual_path:
                self.samples.append((actual_path, item))
                
        print(f"[*] ĐÃ KHỚP THÀNH CÔNG: {len(self.samples)} ảnh và nhãn cho tập [{split_name}]!")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, item = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        
        annotations = []
        if 'labels' in item and item['labels']:
            for obj in item['labels']:
                if obj.get('category') == TARGET_CLASS and 'box2d' in obj:
                    b = obj['box2d']
                    w = max(0.0, float(b['x2'] - b['x1']))
                    h = max(0.0, float(b['y2'] - b['y1']))
                    if w > 1.0 and h > 1.0:
                        annotations.append({
                            "bbox": [float(b['x1']), float(b['y1']), w, h],
                            "category_id": COCO_CAR_ID,
                            "area": w * h,
                            "iscrowd": 0
                        })
        
        target = {"image_id": idx, "annotations": annotations}
        encoding = self.processor(images=image, annotations=target, return_tensors="pt")
        return {k: v[0] for k, v in encoding.items()}

def collate_fn(batch):
    pixel_values = torch.stack([item["pixel_values"] for item in batch])
    labels = [item["labels"] for item in batch]
    return {"pixel_values": pixel_values, "labels": labels}

In [ ]:
# ==============================================================================
# CELL 4: KHỞI TẠO MÔ HÌNH RT-DETR TỪ HUGGING FACE
# ==============================================================================
from transformers import RTDetrForObjectDetection, RTDetrImageProcessor, TrainingArguments, Trainer

print("[*] Đang tải trọng số tiền huấn luyện RT-DETR từ Hugging Face...")
processor = RTDetrImageProcessor.from_pretrained("PekingU/rtdetr_r50vd")
model = RTDetrForObjectDetection.from_pretrained("PekingU/rtdetr_r50vd", ignore_mismatched_sizes=True)

train_dataset = BDD100kDataset(TRAIN_JSON, TRAIN_IMG_DIR, split_name="TRAIN", processor=processor)
val_dataset = BDD100kDataset(VAL_JSON, VAL_IMG_DIR, split_name="VAL", processor=processor)
print("[*] Đã sẵn sàng bước vào Huấn luyện!")

In [ ]:
# ==============================================================================
# CELL 5: TIẾN HÀNH HUẤN LUYỆN VÀ LƯU FILE TRỌNG SỐ (WEIGHTS)
# ==============================================================================
output_dir = "/kaggle/working"
os.makedirs(output_dir, exist_ok=True)

# Cấu hình tham số tối ưu cho GPU T4 (16GB VRAM) của Kaggle
training_args = TrainingArguments(
    output_dir=os.path.join(output_dir, f"checkpoints_expert_{ENSEMBLE_ID}"),
    num_train_epochs=5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,    # Batch size hiệu dụng = 8
    learning_rate=5e-5,
    weight_decay=1e-4,
    save_strategy="epoch",
    logging_steps=50,
    dataloader_num_workers=2,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn
)

print("=" * 60)
print(f"[*] BẮT ĐẦU HUẤN LUYỆN CHUYÊN GIA {ENSEMBLE_ID} (SEED = {CURRENT_SEED})...")
print("=" * 60)

trainer.train()

# Lưu trực tiếp file .pth vào thư mục Output của Kaggle
save_path = os.path.join(output_dir, f"expert_{ENSEMBLE_ID}.pth")
torch.save(model.state_dict(), save_path)

print("=" * 60)
print(f"[*] HUẤN LUYỆN THÀNH CÔNG! ĐÃ LƯU FILE: {save_path}")
print(f"[*] HÃY NHÌN SANG CỘT OUTPUT BÊN PHẢI VÀ TẢI FILE 'expert_{ENSEMBLE_ID}.pth' VỀ MÁY!")
print("=" * 60)